In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import spikeinterface as si
import sys 

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT))

EPHYS_DIR = REPO_ROOT / "DATA" / "ephys"

OUTPUT_DIR = REPO_ROOT / "DATA"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEHAVIOR_FPS = 30.02
ANALYSIS_WINDOW_S = 3.0

TRUNCATION_TOLERANCE_SAMPLES = 1
SYNC_EDGE_MARGIN_S = 0.5


OUTPUT_FILE = OUTPUT_DIR / "analysis_windows.csv"
OUTPUT_FILE_FULL = OUTPUT_DIR / "analysis_windows_full.csv"
SYNC_QC_FILE = OUTPUT_DIR / "sync_qc_summary.csv"

In [ ]:
# Identify recordings containing the processed LFP data
recording_dirs = sorted(
    {
        path.parent.parent
        for path in EPHYS_DIR.rglob("lfp/meta.json")
    }
)

print(f"Found {len(recording_dirs)} recordings.")

In [ ]:
from utils.io import read_csv_auto

# Match behavioral exploration segments to their corresponding ephys samples
# using the LED synchronization frame, then define 3-s analysis windows

analysis_windows = []
sync_qc = []

for recording_dir in recording_dirs:

    recording_name = recording_dir.name
    print(f"Processing: {recording_name}")

    lfp_path = recording_dir / "lfp"
    recording = si.load(lfp_path)

    sampling_frequency = float(
        recording.get_sampling_frequency()
    )

    n_samples = recording.get_num_frames()
    ephys_duration_s = n_samples / sampling_frequency

    metadata_path = lfp_path / "meta.json"

    with metadata_path.open("r") as file:
        metadata = json.load(file)

    led_path = recording_dir / "LED_info.csv"
    segments_path = recording_dir / "exploration_segments.csv"

    led_data = read_csv_auto(led_path)
    exploration_segments = read_csv_auto(segments_path)

    frame_offset = int(led_data["Frame"].iloc[0])

    sync_meta = metadata.get("synchronization", {})
    ephys_sync_channel = sync_meta.get("channel")
    ephys_sync_threshold = sync_meta.get("threshold")
    ephys_sync_start_time_s = sync_meta.get("start_time_s")

    behavior_sync_time_s = frame_offset / BEHAVIOR_FPS

    flag_ephys_sync_suspect = (
        ephys_sync_start_time_s is None
        or ephys_sync_start_time_s < SYNC_EDGE_MARGIN_S
        or ephys_sync_start_time_s > (ephys_duration_s - SYNC_EDGE_MARGIN_S)
    )

    flag_behavior_sync_suspect = (
        frame_offset <= 0
        or behavior_sync_time_s < SYNC_EDGE_MARGIN_S
    )

    if flag_ephys_sync_suspect:
        print(
            f"  WARNING: ephys-side sync time for {recording_name} "
            f"({ephys_sync_start_time_s} s) is close to the recording edge "
            f"(duration {ephys_duration_s:.2f} s) - verify manually."
        )

    if flag_behavior_sync_suspect:
        print(
            f"  WARNING: behavior-side sync frame for {recording_name} "
            f"(frame {frame_offset}, {behavior_sync_time_s:.2f} s) looks "
            f"suspicious - verify manually."
        )

    sync_qc.append(
        {
            "recording": recording_name,
            "ephys_sync_channel": ephys_sync_channel,
            "ephys_sync_threshold": ephys_sync_threshold,
            "ephys_sync_start_time_s": ephys_sync_start_time_s,
            "ephys_duration_s": ephys_duration_s,
            "behavior_sync_frame_offset": frame_offset,
            "behavior_sync_time_s": behavior_sync_time_s,
            "flag_ephys_sync_suspect": flag_ephys_sync_suspect,
            "flag_behavior_sync_suspect": flag_behavior_sync_suspect,
        }
    )

    segments = exploration_segments.copy()

    segments["start_frame"] -= frame_offset
    segments["end_frame"] -= frame_offset

    segments = segments[
        segments["start_frame"] >= 0
    ].copy()

    half_window_frames = int(
        round(BEHAVIOR_FPS * ANALYSIS_WINDOW_S / 2)
    )

    segment_midpoints = (
        segments["start_frame"] +
        segments["end_frame"]
    ) / 2

    segments["window_start_frame"] = (
        segment_midpoints - half_window_frames
    )

    segments["window_end_frame"] = (
        segment_midpoints + half_window_frames
    )

    segments["sample_start"] = (
        segments["window_start_frame"]
        / BEHAVIOR_FPS
        * sampling_frequency
    ).astype(int)

    segments["sample_end"] = (
        segments["window_end_frame"]
        / BEHAVIOR_FPS
        * sampling_frequency
    ).astype(int)

    expected_window_samples = int(
        round(2 * half_window_frames / BEHAVIOR_FPS * sampling_frequency)
    )

    segments["sample_start"] = (
        segments["sample_start"]
        .clip(lower=0, upper=n_samples)
    )

    segments["sample_end"] = (
        segments["sample_end"]
        .clip(lower=0, upper=n_samples)
    )

    actual_window_samples = (
        segments["sample_end"] - segments["sample_start"]
    )

    segments["expected_window_samples"] = expected_window_samples
    segments["actual_window_samples"] = actual_window_samples
    segments["window_truncated"] = (
        actual_window_samples
        < (expected_window_samples - TRUNCATION_TOLERANCE_SAMPLES)
    )

    n_truncated = int(segments["window_truncated"].sum())
    if n_truncated > 0:
        print(
            f"  {n_truncated} / {len(segments)} windows truncated at the "
            f"recording edge for {recording_name} and will be excluded "
            f"from analysis_windows.csv (kept, flagged, in the _full export)."
        )

    for segment_index, segment in segments.iterrows():

        analysis_windows.append(
            {
                "recording": recording_name,
                "segment_index": segment_index,
                "label": segment["label"],
                "behavior_start_frame": segment["start_frame"],
                "behavior_end_frame": segment["end_frame"],
                "sample_start": segment["sample_start"],
                "sample_end": segment["sample_end"],
                "sampling_frequency_hz": sampling_frequency,
                "behavior_fps": BEHAVIOR_FPS,
                "window_duration_s": ANALYSIS_WINDOW_S,
                "expected_window_samples": segment["expected_window_samples"],
                "actual_window_samples": segment["actual_window_samples"],
                "window_truncated": segment["window_truncated"],
            }
        )

In [ ]:
# Assemble the synchronized analysis windows into the final output table

analysis_windows_df = pd.DataFrame(analysis_windows)

column_order = [
    "recording",
    "segment_index",
    "label",
    "behavior_start_frame",
    "behavior_end_frame",
    "sample_start",
    "sample_end",
    "sampling_frequency_hz",
    "behavior_fps",
    "window_duration_s",
    "expected_window_samples",
    "actual_window_samples",
    "window_truncated",
]

analysis_windows_df = analysis_windows_df[column_order]

n_total = len(analysis_windows_df)
n_truncated_total = int(analysis_windows_df["window_truncated"].sum())

analysis_windows_full_df = analysis_windows_df.copy()

analysis_windows_df = analysis_windows_df[
    ~analysis_windows_df["window_truncated"]
].drop(columns=["expected_window_samples", "actual_window_samples", "window_truncated"])

print(
    f"Generated {n_total} candidate windows from "
    f"{analysis_windows_full_df['recording'].nunique()} recordings."
)
print(
    f"Excluded {n_truncated_total} truncated windows "
    f"({n_truncated_total / n_total:.1%}). "
    f"Kept {len(analysis_windows_df)} full-length windows."
)

In [ ]:
# Sync QC summary

sync_qc_df = pd.DataFrame(sync_qc)

n_suspect = int(
    (sync_qc_df["flag_ephys_sync_suspect"] | sync_qc_df["flag_behavior_sync_suspect"]).sum()
)

if n_suspect > 0:
    print(
        f"WARNING: {n_suspect} / {len(sync_qc_df)} recordings have a "
        f"suspect ephys or behavior sync event - review sync_qc_summary.csv "
        f"before trusting their analysis windows."
    )

sync_qc_df

In [ ]:
analysis_windows_df.to_csv(OUTPUT_FILE, index=False)
analysis_windows_full_df.to_csv(OUTPUT_FILE_FULL, index=False)
sync_qc_df.to_csv(SYNC_QC_FILE, index=False)

print(f"Saved final (filtered) dataset to:\n{OUTPUT_FILE}")
print(f"Saved full (unfiltered, flagged) dataset to:\n{OUTPUT_FILE_FULL}")
print(f"Saved sync QC summary to:\n{SYNC_QC_FILE}")